<a href="https://colab.research.google.com/github/Graizelle/sdxl_trainer_pro/blob/main/Graizelle's_SDXL_Trainer_Pro_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚙️ Configure Training & Repos

In [ ]:
# @title Hugging Face & Training Configuration
# @markdown ### Repository Settings
DATASET_REPO = "graizelle/braces" # @param {type:"string"}
MODEL_REPO_ID = "graizelle/graizelle_lustmix_v2.0" # @param {type:"string"}
LORA_REPO_ID = "graizelle/braces_v1" # @param {type:"string"}


# @markdown ### Training Hyperparameters
OUTPUT_NAME = "braces_v1" # @param {type:"string"}
TRIGGER_TOKEN = "braces" # @param {type:"string"}
MAX_TRAIN_STEPS = 800 # @param {type:"integer"}
BATCH_SIZE = 1 # @param {type:"slider", min:1, max:16, step:1}
REPEATS = 4 # @param {type:"slider", min:1, max:100, step:1}
GRADIENT_ACCUMULATION_STEPS = 4 # @param {type:"slider", min:1, max:16, step:1}
RESOLUTION = 1024 # @param {type:"integer"}
NETWORK_DIM = 64 # @param {type:"integer"}
NETWORK_ALPHA = 32 # @param {type:"integer"}
LEARNING_RATE = 0.0004 # @param {type:"number"}
LR_SCHEDULER = "cosine" # @param ["constant", "cosine", "linear", "polynomial", "constant_with_warmup"] {type:"string"}
LR_WARMUP_STEPS = 100 # @param {type:"integer"}
SEED = 42 # @param {type:"integer"}
SAVE_STATE_EVERY = 100 # @param {type:"integer"}
RESUME_FROM_CHECKPOINT = True # @param {type:"boolean"}
CHECKPOINT_FOLDER = "/content/output/braces_v1-step00000600.safetensors" # @param {type:"string"}
OPTIMIZER_TYPE = "Adafactor" # @param ["AdamW", "AdamW8bit", "Adafactor", "DAdaptation", "SGDNesterov", "DAdaptAdam", "DAdaptLion", "DAdaptSGD", "Lion", "Prodigy"] {type:"string"}
OPTIMIZER_ARGS = "scale_parameter=False relative_step=False warmup_init=False" # @param {type:"string"}


print(f"Setup complete for: {OUTPUT_NAME}")
print(f"\n--- Training Configuration Summary ---")
effective_batch_size = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
print(f"Effective Batch Size: {effective_batch_size}")
print(f"Total Optimization Steps: {MAX_TRAIN_STEPS}")
print(f"Total Effective Samples Processed: {effective_batch_size * MAX_TRAIN_STEPS}")
print(f"(Note: 'REPEATS' ({REPEATS}x) multiplies each unique image in your dataset for training, effectively increasing its size.)")

# 🌐 Setup Environment

In [ ]:
# @title Install Dependencies
import os, shutil, subprocess
from huggingface_hub import login
from accelerate.utils import write_basic_config
from google.colab import userdata # Import userdata to access Colab secrets

# Ensure we start in a known good working directory
os.chdir('/content')
print(f"Current working directory set to: {os.getcwd()}")

# Install generic dependencies
!pip install -q huggingface_hub onnxruntime pandas tqdm timm pillow numpy accelerate matplotlib

# Remove old sd-scripts
sd_scripts_path = "/content/sd-scripts"
if os.path.exists(sd_scripts_path):
    print(f"Attempting to remove existing '{sd_scripts_path}'...")
    try:
        shutil.rmtree(sd_scripts_path)
        print(f"Successfully removed '{sd_scripts_path}'.")
    except OSError as e:
        print(f"Error removing '{sd_scripts_path}': {e}")
        print("This might prevent git clone from succeeding. Please manually inspect or clear '/content/sd-scripts'.")
        raise # Re-raise the exception to stop execution if cleanup failed

# After attempted removal, verify it's gone.
if os.path.exists(sd_scripts_path):
    print(f"CRITICAL WARNING: '{sd_scripts_path}' still exists after removal attempt.")
    print("This indicates a deeper issue, and git clone will likely fail.")
    raise FileExistsError(f"Failed to clear directory '{sd_scripts_path}' for cloning.")

# Clone sd-scripts
print("Cloning sd-scripts repository...")
subprocess.run(
    ["git", "clone", "https://github.com/kohya-ss/sd-scripts.git", sd_scripts_path],
    check=True
)
print("sd-scripts cloned successfully.")

# Change working directory to sd_scripts_path for subsequent installations
os.chdir(sd_scripts_path)
print(f"Current working directory changed to: {os.getcwd()}")

# Install PyTorch first (FP16 + CUDA compatible)
# !pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124

# Install remaining requirements
!pip install  -r requirements.txt

# Accelerate basic config
write_basic_config()


# ✅ Set Global Config

In [ ]:
# @title Global Config
# @markdown This cell centralizes all key configurable variables for your project.
# @markdown Run this cell once, and all other cells will reference these settings.

# --- Hugging Face & Repository Settings ---
DATASET_REPO = "graizelle/braces" # @param {type:"string"}
MODEL_REPO_ID = "graizelle/graizelle_lustmix_v2.0" # @param {type:"string"}
LORA_REPO_ID = "graizelle/braces_v1" # @param {type:"string"}

# --- Training Hyperparameters ---
OUTPUT_NAME = "braces_v1" # @param {type:"string"}
TRIGGER_TOKEN = "braces" # @param {type:"string"}
MAX_TRAIN_STEPS = 800 # @param {type:"integer"}
BATCH_SIZE = 1 # @param {type:"slider", min:1, max:16, step:1}
REPEATS = 4 # @param {type:"slider", min:1, max:100, step:1}
GRADIENT_ACCUMULATION_STEPS = 4 # @param {type:"slider", min:1, max:16, step:1}
RESOLUTION = 1024 # @param {type:"integer"}
NETWORK_DIM = 64 # @param {type:"integer"}
NETWORK_ALPHA = 32 # @param {type:"integer"}
LEARNING_RATE = 0.0004 # @param {type:"number"}
LR_SCHEDULER = "cosine" # @param ["constant", "cosine", "linear", "polynomial", "constant_with_warmup"] {type:"string"}
LR_WARMUP_STEPS = 100 # @param {type:"integer"}
SEED = 42 # @param {type:"integer"}
SAVE_STATE_EVERY = 100 # @param {type:"integer"}
RESUME_FROM_CHECKPOINT = False # @param {type:"boolean"}
CHECKPOINT_FOLDER = "" # @param {type:"string"}
OPTIMIZER_TYPE = "Adafactor" # @param ["AdamW", "AdamW8bit", "Adafactor", "DAdaptation", "SGDNesterov", "DAdaptAdam", "DAdaptLion", "DAdaptSGD", "Lion", "Prodigy"] {type:"string"}
OPTIMIZER_ARGS = "scale_parameter=False relative_step=False warmup_init=False" # @param {type:"string"}
RUN_TRAINING_IN_BACKGROUND = False # @param {type:"boolean"}

# --- Dataset & Tagging Configuration ---
OVERWRITE_EXISTING = False # @param {type:"boolean"}
TAG_THRESHOLD = 0.64 # @param {type:"slider", min:0.0, max:1.0, step:0.01}
SEARCH_TERM = "" # @param {type:"string"}
REPLACE_WITH = "" # @param {type:"string"}
ADD_TAGS = "" # @param {type:"string"}
REMOVE_TAGS = "1boy, 1girl" # @param {type:"string"}
TOP_TAGS = 100 # @param {type:"integer"}
SHOW_CHART = True # @param {type:"boolean"}

# --- Test & Upload Configuration ---
PROMPT = "1girl, deelite, solo, masterpiece, cinematic lighting" # @param {type:"string"}
NEGATIVE_PROMPT = "low quality, bad anatomy, text, watermark, blurry" # @param {type:"string"}
STEPS = 35 # @param {type:"slider", min:20, max:50, step:1}
CFG_SCALE = 2.2 # @param {type:"number"}
LORA_STRENGTH = 0.7 # @param {type:"slider", min:0.1, max:1.5, step:0.1}
COMMIT_MESSAGE = "Updated dataset with new tags and deduplication" # @param {type:"string"}

print(f"Global Configuration setup complete for project: {OUTPUT_NAME}")
print(f"All subsequent cells will use these values.")

# 🤗 Huggingface

In [ ]:
# @title Huggingface login
!pip install -q huggingface_hub

from huggingface_hub import login
from google.colab import userdata

try:
    # Try to get the token from Colab secrets
    TOKEN = userdata.get('HF_TOKEN')

    if TOKEN:
        print("✅ Loaded HF_TOKEN from secrets")
        # Force overwrite any broken cached token - HfFolder is no longer needed
        login(token=TOKEN, add_to_git_credential=True)
        print("✅ HuggingFace login successful")
    else:
        raise ValueError("HF_TOKEN not found in secrets")

except Exception as e:
    print("❌ Secret login failed:", e)
    print("🔐 Falling back to manual login...")
    login()  # Manual login prompt

In [ ]:
# @title Download Checkpoint from Huggingface Repo
from huggingface_hub import HfApi, hf_hub_download
import os

LOCAL_MODEL_DIR = "/content/models"
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

api = HfApi()
files = api.list_repo_files(MODEL_REPO_ID)

# Find first .safetensors file
safetensors_files = [f for f in files if f.endswith(".safetensors")]
if not safetensors_files:
    raise ValueError(f"No .safetensors files found in {MODEL_REPO_ID}")

checkpoint_filename = safetensors_files[0]
print(f"✅ Found checkpoint: {checkpoint_filename}")

# ✅ IMPORTANT: use local_dir instead of cache_dir
MODEL_PATH = hf_hub_download(
    repo_id=MODEL_REPO_ID,
    filename=checkpoint_filename,
    local_dir=LOCAL_MODEL_DIR,
)

print(f"✅ Checkpoint downloaded to: {MODEL_PATH}")

# ✅ Verify file size (CRITICAL)
size_gb = os.path.getsize(MODEL_PATH) / (1024**3)
print(f"📦 File size: {size_gb:.2f} GB")

if size_gb < 1:
    raise ValueError("❌ Downloaded file is too small — likely a pointer file!")

In [ ]:
# @title Auto-detect + convert SDXL model if needed

from safetensors.torch import load_file
import os

MODEL_INPUT = MODEL_PATH  # your existing variable
MODEL_OUTPUT = MODEL_INPUT.replace(".safetensors", "-converted.safetensors")

def is_sdxl_checkpoint(sd):
    keys = list(sd.keys())
    return any(k.startswith("model.diffusion_model") for k in keys)

def is_diffusers_format(sd):
    keys = list(sd.keys())
    return any("conditioner.embedders" in k for k in keys)

print(f"🔎 Checking model format: {MODEL_INPUT}")

sd = load_file(MODEL_INPUT)

if is_sdxl_checkpoint(sd):
    print("✅ Model is already in SDXL checkpoint format. No conversion needed.")
    FINAL_MODEL_PATH = MODEL_INPUT

elif is_diffusers_format(sd):
    print("⚠️ Diffusers/SGM format detected. Converting...")

    if not os.path.exists(MODEL_OUTPUT):
        !python /content/sd-scripts/tools/convert_diffusers_to_original_sdxl.py \
          --model_path "{MODEL_INPUT}" \
          --checkpoint_path "{MODEL_OUTPUT}" \
          --half
    else:
        print("ℹ️ Converted model already exists, skipping conversion.")

    FINAL_MODEL_PATH = MODEL_OUTPUT

else:
    raise ValueError("❌ Unknown model format. Cannot determine how to proceed.")

print(f"\n🎯 Using model: {FINAL_MODEL_PATH}")

# 📊 Dataset & Tagging

In [ ]:
# @title Download & Prepare Dataset from Huggingface Repo
import os, shutil # Added import for os and shutil

# --- CONFIG SYNC ---
repo_to_use = DATASET_REPO
char_to_use = OUTPUT_NAME.split('_')[0]

# --- PATH LOGIC ---
TEMP_DIR = "/content/temp_dataset"
BASE_DATASET_DIR = "/content/dataset"
FOLDER_NAME = f"{REPEATS}_{char_to_use}"
FINAL_DIR = os.path.join(BASE_DATASET_DIR, FOLDER_NAME)

# ✅ THIS is the key line
DATA_DIR = "/content/dataset"

# --- CLEAN ---
if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)
if os.path.exists(FINAL_DIR):
    shutil.rmtree(FINAL_DIR)

os.makedirs(FINAL_DIR, exist_ok=True);

print(f" Syncing from HF Root: {repo_to_use}")
print(f" Creating Trainer Structure: {FINAL_DIR}")

# Now import snapshot_download after settings are in place
from huggingface_hub import snapshot_download


# --- DOWNLOAD ---
try:
    snapshot_download(
        repo_id=repo_to_use,
        repo_type="dataset",
        local_dir=TEMP_DIR,
        token=True
    )

    # Flatten files into FINAL_DIR
    for root, dirs, files in os.walk(TEMP_DIR):
        for f in files:
            if f.startswith('.'):
                continue

            shutil.move(
                os.path.join(root, f),
                os.path.join(FINAL_DIR, f)
            )

    shutil.rmtree(TEMP_DIR)

except Exception as e:
    print(f"\n❂ Error during download: {e}")

# --- SANITY CHECK ---
print(f"✅ DATA_DIR set to: {DATA_DIR}")
print(f"Subfolders: {os.listdir(DATA_DIR)}")
print(f"Files in subset: {len(os.listdir(FINAL_DIR))}")

In [ ]:
# @title Manual Exact-Duplicate Remover
import os, hashlib
from tqdm.notebook import tqdm

# Update to your confirmed path
DATASET_PATH = FINAL_DIR # Use FINAL_DIR from Cell 3
# NUM_IMAGES_DEDUP was not being used and has been removed.
def get_file_hash(filepath):
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        hasher.update(f.read())
    return hasher.hexdigest()

if not os.path.exists(DATASET_PATH):
    print(f"❌ ERROR: Folder {DATASET_PATH} not found.")
elif not os.listdir(DATASET_PATH):
    print(f"⚠️ Warning: Folder {DATASET_PATH} is empty. No images to deduplicate.")
else:
    print("🚀 Removing Exact Duplicates...")
    files = [f for f in os.listdir(DATASET_PATH) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    hashes = {}
    duplicates_removed = 0

    for filename in tqdm(files, desc="Deduplicating"): # Added tqdm
        file_path = os.path.join(DATASET_PATH, filename)
        file_hash = get_file_hash(file_path)
        if file_hash in hashes:
            os.remove(file_path)
            duplicates_removed += 1
        else:
            hashes[file_hash] = filename

    print(f"✅ Done! Removed {duplicates_removed} duplicates. {len(hashes)} unique images remain.")

In [ ]:
# @title WD14 Auto-Tagging
# @markdown This cell will NOT overwrite existing .txt files unless checked.

import os, torch, numpy as np, timm, requests
from PIL import Image
from tqdm.notebook import tqdm
from IPython.display import clear_output

# --- SETTINGS ---
DATASET_PATH = FINAL_DIR # Use FINAL_DIR from Cell 3
OVERWRITE_EXISTING = False # @param {type:"boolean"}
TAG_THRESHOLD = 0.64 # @param {type:"slider", min:0.0, max:1.0, step:0.01}
TRIGGER_TOKEN = TRIGGER_TOKEN # Use TRIGGER_TOKEN from Cell 1

if not os.path.exists(DATASET_PATH):
    print(f"❌ ERROR: Folder {DATASET_PATH} not found.")
elif not os.listdir(DATASET_PATH):
    print(f"⚠️ Warning: Folder {DATASET_PATH} is empty. No images to tag.")
else:
    print("🚀 Loading Tagger Model...")
    model = timm.create_model("hf_hub:SmilingWolf/wd-vit-tagger-v3", pretrained=True).eval()
    if torch.cuda.is_available(): model = model.cuda()

    tag_url = "https://huggingface.co/SmilingWolf/wd-v1-4-vit-tagger/resolve/main/selected_tags.csv"
    tag_names = [line.split(",")[1] for line in requests.get(tag_url).text.splitlines()[1:]]

    files = [f for f in os.listdir(DATASET_PATH) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    if not files:
        print("No image files found in the dataset folder to tag.")
    else:
        tagged_count = 0
        skipped_count = 0

        print(f"Processing {len(files)} images...")
        for i, file in enumerate(tqdm(files, desc="Tagging")): # Added tqdm
            path = os.path.join(DATASET_PATH, file)
            txt_path = path.rsplit(".", 1)[0] + ".txt"

            # Check if we should skip
            if os.path.exists(txt_path) and not OVERWRITE_EXISTING:
                skipped_count += 1
                continue

            try:
                img = Image.open(path).convert("RGB").resize((448, 448))
                img_tensor = torch.tensor(np.transpose(np.array(img).astype(np.float32) / 255.0, (2, 0, 1))).unsqueeze(0)
                if torch.cuda.is_available(): img_tensor = img_tensor.cuda()

                with torch.no_grad():
                    preds = model(img_tensor)[0].cpu().numpy()

                found_tags = [tag_names[idx].replace("_", " ") for idx, p in enumerate(preds) if p > TAG_THRESHOLD and "rating" not in tag_names[idx]]
                final_tags = f"{TRIGGER_TOKEN}, " + ", ".join(found_tags)

                with open(txt_path, "w") as f:
                    f.write(final_tags)
                tagged_count += 1

                # Removed clear_output here as it clears the tqdm bar
            except Exception as e:
                print(f"Error on {file}: {e}")
                continue

        clear_output() # Clears tqdm output only once at the end
        print(f"✅ COMPLETE!")
        print(f"New Tags Created: {tagged_count}")
        print(f"Existing Files Preserved: {skipped_count}")

In [ ]:
# @title Advanced Tag Editing (Search, Replace, Add)
# @markdown Run this cell after auto-tagging to refine your dataset.
SEARCH_TERM = "" # @param {type:"string"}
REPLACE_WITH = "" # @param {type:"string"}
ADD_TAGS = "" # @param {type:"string"}
REMOVE_TAGS = "1boy, 1girl" # @param {type:"string"}

import os

DATASET_PATH = FINAL_DIR # Use FINAL_DIR from Cell 3
remove_list = [t.strip() for t in REMOVE_TAGS.split(",")]

for file in os.listdir(DATASET_PATH):
    if file.endswith(".txt"):
        path = os.path.join(DATASET_PATH, file)
        with open(path, "r") as f:
            tags = f.read()

        # 1. Bulk Search and Replace
        if SEARCH_TERM:
            tags = tags.replace(SEARCH_TERM, REPLACE_WITH)

        # 2. Remove specific bad tags
        for bad in remove_list:
            tags = tags.replace(bad, "")

        # 3. Append new global tags
        if ADD_TAGS:
            tags = tags + ", " + ADD_TAGS

        # Cleanup extra commas/spaces
        tags = ", ".join([t.strip() for t in tags.split(",") if t.strip()])

        with open(path, "w") as f:
            f.write(tags)
print("Tag editing complete.")

In [ ]:
import os
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
# @title Analyze Tags
# --- SETTINGS ---
TOP_TAGS = 100 # @param {type:"integer"}
SHOW_CHART = True # @param {type:"boolean"}

# Re-defining FINAL_DIR for robustness against kernel restarts.
# These variables (OUTPUT_NAME, REPEATS) should be globally available if Cell 1 ('Configure Training & Repos') was run.
# BASE_DATASET_DIR is a constant path.
BASE_DATASET_DIR = "/content/dataset"
char_to_use = OUTPUT_NAME.split('_')[0]
FOLDER_NAME = f"{REPEATS}_{char_to_use}"
FINAL_DIR = os.path.join(BASE_DATASET_DIR, FOLDER_NAME)

# Pointing to your specific nested folder
dataset_path = FINAL_DIR # Use FINAL_DIR from Cell 3
tag_counts = Counter()
total_files = 0

if not os.path.exists(dataset_path):
    print(f"❌ ERROR: Folder not found at {dataset_path}. Check Step 3.")
elif not os.listdir(dataset_path):
    print(f"⚠️ Warning: Folder {dataset_path} is empty. No images to analyze.")
else:
    # Parse all .txt files
    for file in os.listdir(dataset_path):
        if file.endswith(".txt"):
            total_files += 1
            with open(os.path.join(dataset_path, file), "r") as f:
                # Splitting by comma and cleaning whitespace
                tags = [t.strip() for t in f.read().split(",") if t.strip()]
                tag_counts.update(tags)

    if total_files == 0:
        print("No .txt files found. Did you run the auto-tagging cell?")
    else:
        # Create DataFrame for analysis
        df_tags = pd.DataFrame(tag_counts.most_common(), columns=['Tag', 'Count'])
        df_tags['Percentage'] = (df_tags['Count'] / total_files * 100).round(2)

        # Verify Trigger Token Presence (Uses TRIGGER_TOKEN from Cell 1)
        trigger_presence = tag_counts.get(TRIGGER_TOKEN, 0)
        trigger_pct = (trigger_presence / total_files * 100)

        print(f"--- Dataset Statistics ---")
        print(f"Total Image/Caption Pairs: {total_files}")
        print(f"Trigger Token ('{TRIGGER_TOKEN}') Presence: {trigger_pct:.2f}%")

        if trigger_pct < 100:
            missing = total_presence = trigger_presence
            print(f"⚠️ WARNING: Trigger token missing in {missing} files!")

        # New calculations for training cycle estimation
        try:
            # These variables are defined in Cell 1 (v8X-CC_0Vl8M)
            # and should be globally accessible after that cell is run.
            effective_batch_size = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
            if effective_batch_size > 0:
                steps_per_effective_epoch = total_files / effective_batch_size
                total_effective_epochs = MAX_TRAIN_STEPS / steps_per_effective_epoch
                print(f"\n--- Training Cycle Estimation ---")
                print(f"Effective Batch Size: {effective_batch_size}")
                print(f"Steps per Effective Epoch: {steps_per_effective_epoch:.2f}")
                print(f"Total Effective Epochs (based on MAX_TRAIN_STEPS={MAX_TRAIN_STEPS}): {total_effective_epochs:.2f}")
            else:
                print("\n⚠️ Cannot calculate training cycle estimation: Effective Batch Size is zero.")
        except NameError:
            print("\n⚠️ Cannot calculate training cycle estimation: Training hyperparameters (BATCH_SIZE, GRADIENT_ACCUMULATION_STEPS, MAX_TRAIN_STEPS) not defined. Please run Cell 1.")
        except ZeroDivisionError:
            print("\n⚠️ Cannot calculate training cycle estimation: Steps per effective epoch is zero, possibly due to zero effective batch size or no image files.")


        print(f"--------------------------\n")

        print(f"Top {TOP_TAGS} Tags:")
        # We use head(TOP_TAGS + 1) in case the trigger token is #1
        # to ensure you see enough descriptive tags.
        print(df_tags.head(TOP_TAGS).to_string(index=False))

        if SHOW_CHART:
            # Clear previous plots to save browser memory
            plt.close('all')
            plt.figure(figsize=(10, TOP_TAGS * 0.15)) # Dynamically adjust height based on TOP_TAGS
            top_df = df_tags.head(TOP_TAGS)

            # Create horizontal bar chart
            plt.barh(top_df['Tag'][::-1], top_df['Count'][::-1], color='#4A90E2')
            plt.xlabel('Number of Images')
            plt.title(f'Top {TOP_TAGS} Most Frequent Tags in {OUTPUT_NAME}')
            plt.grid(axis='x', linestyle='--', alpha=0.7)
            plt.tight_layout()
            plt.show()

# ⚡️ Training


In [ ]:
# @title
import os
import sys

os.environ["PYTHONPATH"] = "/content/sd-scripts"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTHONUNBUFFERED"] = "1" # Set PYTHONUNBUFFERED as an environment variable

# @markdown ### Training Execution Settings
RUN_TRAINING_IN_BACKGROUND = True # @param {type:"boolean"}

# If your OPTIMIZER_ARGS variable was "scale_parameter=False relative_step=False warmup_init=False"
# We need to make sure they are passed without the wrapping quotes in the final command.

training_command = f"""
accelerate launch --num_processes=1 --mixed_precision="fp16" \
  /content/sd-scripts/sdxl_train_network.py \
  --lowram \
  --pretrained_model_name_or_path="{FINAL_MODEL_PATH}" \
  --caption_extension=".txt" \
  --resolution={RESOLUTION},{RESOLUTION} \
  --output_dir=/content/output \
  --output_name={OUTPUT_NAME} \
  --train_data_dir=/content/dataset \
  --enable_bucket \
  --flip_aug \
  --network_module=networks.lora \
  --network_dim={NETWORK_DIM} \
  --network_alpha={NETWORK_ALPHA} \
  --learning_rate={LEARNING_RATE} \
  --lr_scheduler={LR_SCHEDULER} \
  --lr_warmup_steps={LR_WARMUP_STEPS} \
  --train_batch_size={BATCH_SIZE} \
  --gradient_accumulation_steps={GRADIENT_ACCUMULATION_STEPS} \
  --max_train_steps={MAX_TRAIN_STEPS} \
  --mixed_precision=fp16 \
  --save_precision=fp16 \
  --seed={SEED} \
  --gradient_checkpointing \
  --cache_latents \
  --cache_latents_to_disk \
  --vae_batch_size=1 \
  --optimizer_type={OPTIMIZER_TYPE} \
  --optimizer_args {OPTIMIZER_ARGS} \
  --max_data_loader_n_workers=1 \
  --logging_dir=/content/output/logs \
  --log_with=tensorboard \
  --save_every_n_steps={SAVE_STATE_EVERY} \
  --save_last_n_steps=5 \
  --log_prefix={OUTPUT_NAME} \
  --sdpa \
  --persistent_data_loader_workers \
  --max_token_length=225
""".strip()

if RUN_TRAINING_IN_BACKGROUND:
  print("🚀 Starting training in the background... run tail or tensorview cell for progress!")
  !nohup {training_command} > /content/training.log 2>&1 &
else:
  print("🚀 Starting training directly in the foreground. This cell will block until training completes or is interrupted.")
  !{training_command} 2>&1 | tee /content/training.log

# 🚀 Test & Upload to Huggingface

In [ ]:
# @title 🎨 Test your LoRA
# @markdown Run this AFTER training is 100% and you've Restarted your Session.

PROMPT = "1girl,  solo, masterpiece, cinematic lighting" # @param {type:"string"}
NEGATIVE_PROMPT = "low quality, bad anatomy, text, watermark, blurry" # @param {type:"string"}
STEPS = 35 # @param {type:"slider", min:20, max:50, step:1}
CFG_SCALE = 2.2 # @param {type:"number"}
LORA_STRENGTH = 0.7 # @param {type:"slider", min:0.1, max:1.5, step:0.1}

import torch
import glob
import os
from diffusers import StableDiffusionXLPipeline
from PIL import Image
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import load_file

# --- Re-initialization of FINAL_MODEL_PATH after restart ---
LOCAL_MODEL_DIR = "/content/models"
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

api = HfApi()
files = api.list_repo_files(MODEL_REPO_ID)

safetensors_files = [f for f in files if f.endswith(".safetensors")]
if not safetensors_files:
    raise ValueError(f"No .safetensors files found in {MODEL_REPO_ID}")

checkpoint_filename = safetensors_files[0]
expected_local_path = os.path.join(LOCAL_MODEL_DIR, checkpoint_filename)

# Explicitly check if the model exists locally and is of a reasonable size
# (e.g., > 50MB to avoid partial files from previous failed downloads)
if os.path.exists(expected_local_path) and os.path.getsize(expected_local_path) > 1024 * 1024 * 50:
    MODEL_PATH = expected_local_path
    print(f"✅ Base model found locally at: {MODEL_PATH}")
else:
    print(f"⬇️ Downloading base model: {checkpoint_filename} from {MODEL_REPO_ID}")
    MODEL_PATH = hf_hub_download(
        repo_id=MODEL_REPO_ID,
        filename=checkpoint_filename,
        local_dir=LOCAL_MODEL_DIR,
    )
    print(f"✅ Base model downloaded to: {MODEL_PATH}")

def is_sdxl_checkpoint(sd):
    keys = list(sd.keys())
    return any(k.startswith("model.diffusion_model") for k in keys)

def is_diffusers_format(sd):
    keys = list(sd.keys())
    return any("conditioner.embedders" in k for k in keys)

MODEL_INPUT = MODEL_PATH
MODEL_OUTPUT = MODEL_INPUT.replace(".safetensors", "-converted.safetensors")
sd = load_file(MODEL_INPUT)

if is_sdxl_checkpoint(sd):
    FINAL_MODEL_PATH = MODEL_INPUT
    print("✅ Model is already in SDXL checkpoint format. No conversion needed.")
elif is_diffusers_format(sd):
    print("⚠️ Diffusers/SGM format detected. Checking for conversion...")
    if not os.path.exists(MODEL_OUTPUT):
        # This part requires a shell command, which can't be easily run mid-Python script here
        # For simplicity, we assume if it was converted before, the file exists.
        # If this is a fresh run after restart, the conversion might be skipped
        # if you only re-ran this cell. It's best to run the conversion cell itself.
        # However, to avoid a NameError, we'll set FINAL_MODEL_PATH anyway.
        print("ℹ️ Note: If conversion is truly needed, please run the 'Auto-detect + convert SDXL model' cell above.")
    FINAL_MODEL_PATH = MODEL_OUTPUT
else:
    raise ValueError("❌ Unknown model format. Cannot determine how to proceed.")

print(f"🎯 Using model: {FINAL_MODEL_PATH}")
# --- End re-initialization ---

# 1. Detect the file (since sd-scripts adds step numbers)
lora_files = glob.glob(f"/content/output/{OUTPUT_NAME}*.safetensors")
if not lora_files:
    raise FileNotFoundError("❌ No LoRA file found in /content/output/!")
latest_lora = os.path.basename(max(lora_files, key=os.path.getctime))

print(f"🔄 Loading Base: {FINAL_MODEL_PATH}")
print(f"💉 Injecting LoRA: {latest_lora} at {LORA_STRENGTH} strength")

# Clear VRAM before loading the pipeline for generation
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ Cleared CUDA cache.")

# 2. Load the pipeline with the model you actually trained on (Lustmix)
pipe = StableDiffusionXLPipeline.from_single_file(
    FINAL_MODEL_PATH,
    torch_dtype=torch.float16,
    use_safetensors=True
).to("cuda")

# 3. Apply LoRA with the specific scale
# adapter_name lets us target the specific weights we just loaded
pipe.load_lora_weights("/content/output/", weight_name=latest_lora, adapter_name="deelite")
pipe.set_adapters(["deelite"], adapter_weights=[LORA_STRENGTH])

print("✨ Generating sample...") # Changed resolution
with torch.inference_mode():
    image = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=STEPS,
        guidance_scale=CFG_SCALE,
        width=896, # Reduced width
        height=640 # Reduced height
    ).images[0]

# Show it in the Colab output
display(image)
image.save(f"/content/sample_{OUTPUT_NAME}_{LORA_STRENGTH}.png")
print(f"✅ Saved to /content/sample_{OUTPUT_NAME}_{LORA_STRENGTH}.png")

In [ ]:
# @title 🚀 Upload to Hugging Face
import glob
import os
from huggingface_hub import upload_file

# Look for any .safetensors files in your output folder
lora_files = glob.glob(f"/content/output/*.safetensors")

if lora_files:
    print(f"📦 Found {len(lora_files)} file(s). Starting upload...")
    for file_path in lora_files:
        filename = os.path.basename(file_path)
        print(f"📤 Uploading: {filename} to {LORA_REPO_ID}")

        try:
            upload_file(
                path_or_fileobj=file_path,
                path_in_repo=filename,
                repo_id=LORA_REPO_ID,
            )
            print(f"✅ Successfully uploaded {filename}")
        except Exception as e:
            print(f"❌ Failed to upload {filename}: {e}")

    print("\n✨ All available files have been processed!")
else:
    print("⚠️ No .safetensors files found in /content/output/. Is training still running?")

In [ ]:
# @title 🚀  Upload Dataset to Hugging Face
# @markdown Uploads the processed dataset (images and text files) to your Hugging Face Dataset repo.

from huggingface_hub import upload_folder
import os

# Re-defining FINAL_DIR for robustness against kernel restarts.
# These variables (OUTPUT_NAME, REPEATS) should be globally available if Cell 1 ('Configure Training & Repos') was run.
# BASE_DATASET_DIR is a constant path.
BASE_DATASET_DIR = "/content/dataset"
char_to_use = OUTPUT_NAME.split('_')[0]
FOLDER_NAME = f"{REPEATS}_{char_to_use}"
FINAL_DIR = os.path.join(BASE_DATASET_DIR, FOLDER_NAME)

# Ensure DATASET_REPO is defined (from Cell 1)

if not os.path.exists(FINAL_DIR) or not os.listdir(FINAL_DIR):
    print(f"❌ Error: Dataset directory {FINAL_DIR} is empty or does not exist. Cannot upload.")
else:
    print(f"🚀 Uploading dataset from {FINAL_DIR} to Hugging Face repo: {DATASET_REPO}")


    COMMIT_MESSAGE = "Updated dataset with new tags and deduplication"

    try:
        upload_folder(
            folder_path=FINAL_DIR,
            repo_id=DATASET_REPO,
            repo_type="dataset",
            commit_message=COMMIT_MESSAGE
        )
        print(f"✅ Successfully uploaded dataset to {DATASET_REPO} with commit message: '{COMMIT_MESSAGE}'")
    except Exception as e:
        print(f"❌ Failed to upload dataset: {e}")

# 🛠️ Utilities & Logs

In [ ]:
# @title # 🔧 Live Training Progress
!tail -f /content/training.log

In [ ]:
# @title # 📊 Tensorboard Analysis
import os
from google.colab import output

# Define your logging directory
LOGGING_DIR = os.path.join("/content/output", "logs")

# Check if the directory exists
if not os.path.exists(LOGGING_DIR):
    print(f"⚠️ Warning: '{LOGGING_DIR}' not found. Start training first to generate logs!")
else:
    print(f"🚀 Launching TensorBoard from: {LOGGING_DIR}")

    # 1. Kill any existing instances to prevent port conflicts
    !pkill -f 'tensorboard'

    # 2. Load the extension and launch the dashboard directly in the cell
    %load_ext tensorboard
    %tensorboard --logdir {LOGGING_DIR}

    # 3. Generate a clickable link that works outside the cell (the "Non-Localhost" link)
    print("\n--- Access Links ---")
    output.serve_kernel_port_as_window(6006)

In [ ]:
# @title #🏋️‍♀️ GPU Load
!nvidia-smi